
#The Graph Isomorphism Problem using SMT solvers

Two graphs can be drawn completely differently and still be the same graph underneath,
relabelled. Deciding whether that is the case is the graph isomorphism problem, and it is
one of the few problems we know of that is neither known to be easy nor known to be hard.
In this notebook we will look at SMT solvers and see how to hand the question to Z3: we
will ask it to find a relabelling that preserves every edge, and let it report back when
no such relabelling exists.


**Instructions:**
1. To get started, click on File on the top left and click "Save a copy in Drive."
This will give you an editable version of this document that you can use.
2. If you press `CMD`+`Enter` it runs the cell, and if you press `Shift`+`Enter` it runs the cell and goes to the next one.
3. Make sure you run all cells as you go through the notebook; some cells will not work properly unless the previous one
has been run too.
4. If you disconnect or are inactive for some time you should run all of the cells again.

## 0. Preliminaries (you should run this cell but there is no need to read it)

In [ ]:
!pip install z3-solver
!pip install git+https://github.com/crrivero/FormalMethodsTasting.git#subdirectory=core
from z3 import *
from tofmcore import showSolver, plot_isomorphism
import networkx as nx
import matplotlib.pyplot as plt
from IPython.display import clear_output
clear_output()

## Encoding constraints in Z3

The goal of this notebook is to teach you about formal methods;
particularly, how you can use existing formal verification tools
(in this case, Z3) to analyze and solve your own problems.
Before we get started, let's look at some basic things we can do with Z3.

### Boolean

Suppose you have the following three boolean constraints and you want to check if there's a solution (an assignment of the variables) that satisfies all of them:

$$ x_1 \lor x_2 \lor x_3 $$

$$ \neg x_1 \implies \neg x_2$$

$$  x_1 \land x_3  $$

Let's see how we can do this using Z3.

In [ ]:
s = Solver() # initialize Z3 solver

# initialize variables

x1 = Bool('x_1') # declaring that x_1 is a boolean variable in Z3 which will be referred to as x1 in Python
x2 = Bool('x_2')
x3 = Bool('x_3')

# Note: we can also initialize multiple variables like so: x1, x2, x3 = Bools('x_1 x_2 x_3')

# we use s.add(.) to add a constraint to our solver s
# constraints can be made using many different operations such as Or, And, Not,
# equality, etc.

# here's how we would add the constraints above to our solver:

s.add( Or( x1, x2, x3 ) ) # add the first constraint
s.add( Implies( Not(x1), Not(x2) ) ) # add the second constraint
s.add( And( x1, x3 ) ) # add the third constraint

In [ ]:
# to view the constraints in our solver, we can use the following:
print( s )
# this prints the constraints as they appear in Z3 using Z3's notation

For better readability, this notebook also has a custom print function to view our constraints in LaTeX format, like so:

In [ ]:
showSolver( s )

In [ ]:
# we can use s.check() to run the solver and check whether its satisfiable:
print ( s.check() )

 "sat" means our system of constraints is satisfiable

In [ ]:
# after using s.check(),  we can use s.model() to output a solution if one exsits
solution = s.model()
print( solution )

Let's modify our system of constraint a bit and see if it's still satisfiable. Suppose we want to check if there's a solution where $x_1 = \neg x_3$. Let's see how we would do this with Z3.

In [ ]:
s.add( x1 == Not(x3) )
showSolver( s )

In [ ]:
print( s.check() ) # check if solution exists with new constraint

"unsat" means the system is not satisfiable, i.e., there is no assignment on the variables $x_1$, $x_2$, and $x_3$ that satisfies all the constraints we gave to the solver. **Note that if we were to run s.model() now we would get an error.**

### Integers

Let's use Z3 to solve problems involving integers. Let's start with something simple: find $x$ such that

$$2x + 5 = 15$$

In [ ]:
# Initialize variables

x = Int('x') # declairing that x is an integer named 'x'

# Initialize Z3 solver
s = Solver()

s.add( 2*x + 5 == 15 ) # add the equation

print(s)
print(s.check())
print(s.model())

Now let's try to check whether that's the only solution. We can do this by adding the following constraint to the solver:

$$x \not= 5$$

If the solver returns "**unsat**" then $x=5$ is the only solution.
Try it yourself by completing the code in the cell below.

In [ ]:
s.add( x == 5 ) # REPLACE THIS LINE
s.check()

## The Graph Isomorphism Problem

For two graphs $G$ and $H$, an isomorphism is a function $f$ that maps every node in graph $G$ to a node in graph $H$ such that each edge in $G$, corresponds to an edge in $H$.

More formally:

An isomorphism $ f: V(G) \rightarrow V(H) $ is a bijection such that
$$ (u, v) \in E(G) \iff (f(u), f(v)) \in E(H). $$
We call this function "structure-preserving" or "edge-preserving."
If two graphs have an isomorphism between them, they are said to be isomorphic.

A bijective function $f: X \rightarrow Y$ has the following properties:

$$ \forall x \in X \left( f^{-1}(f(x)) = x \right) $$
$$ \forall y \in Y \left( f(f^{-1}(y)) = y \right) $$

where $f^{-1}$ is the inverse of $f$.

For example, these two graphs are isomorphic:

In [ ]:
G = nx.Graph()
G.add_nodes_from([1, 8])
edges = [(1, 2), (1, 4), (1, 6),
         (2, 3), (2, 5), (3, 4),
         (3, 8), (4, 7), (5, 6),
         (5, 8), (6, 7), (7, 8)]
G.add_edges_from(edges)
G_pos = {1: (0, 3), 2: (1, 3), 3: (0, 2), 4: (1, 2), 5: (0, 1), 6: (1, 1), 7: (0, 0), 8: (1, 0)}

H = nx.Graph()
H.add_nodes_from([1, 8])
edges = [(1, 2), (1, 4), (1, 5),
         (2, 3), (2, 6), (3, 4),
         (3, 7), (4, 8), (5, 6),
         (5, 8), (6, 7), (7, 8)]
H.add_edges_from(edges)
H_pos = {1: (0, 3), 2: (3, 3), 3: (3, 0), 4: (0, 0), 5: (1, 2), 6: (2, 2), 7: (2, 1), 8: (1, 1)}

plt.subplot(1, 2, 1)
nx.draw(G, with_labels=True, pos=G_pos)

plt.subplot(1, 2, 2)
nx.draw(H, with_labels=True, pos=H_pos)

But these two are not:

In [ ]:
G = nx.complete_graph(5)
H = nx.complete_graph(5)
H.remove_edges_from([(0, 1), (1, 2), (2, 3), (3, 4), (4, 0)])

plt.subplot(1, 2, 1)
nx.draw(G, with_labels=True)

plt.subplot(1, 2, 2)
nx.draw_circular(H, with_labels=True)

## Making a Graph Isomorphism Solver

We will use Z3 to find a isomorphism between two graphs $G$ and $H$. We can use Z3's `Function` datatype in order to find a function from nodes in $G$ to nodes in $H$.

However, we will need to restrict the input and output of our function to be limited to nodes in $G$ and nodes in $H$ respectively.
We call this property "closure."

This can be expressed as:

$$ \forall g \in G,  \exists h \in H \left( f(g) = h \right) $$
$$ \forall h \in H, \exists g \in G \left( f^{-1}(h) = g \right) $$

We will use `Or` statements to encode the existential ($\exists$) constraints and for loops to encode the universal ($\forall$) constraints.

In [ ]:
# finds an isomorphism between two undirected graphs
def find_isomorphism(G, H, solver):

    # assumes graph labels are integers
    f = Function('f', IntSort(), IntSort())     # isomorphism f
    f_1 = Function('f_1', IntSort(), IntSort()) # inverse of f (f^(-1))

    for g in G.nodes():
        solver.add(f_1(f(g)) == g)                      # bijection constraint
        solver.add(Or([f(g) == h for h in H.nodes()]))  # closure constraint

    for h in H.nodes():
        solver.add(f(f_1(h)) == h)                          # bijection constraint
        solver.add(Or([f_1(h) == g for g in G.nodes()]))    # closure constraint

    # edge-preserving constraint
    for u_g, v_g in G.edges():
        solver.add(Or([
            Or(And(f(u_g) == u_h, f(v_g) == v_h),   # need to check both directions,
               And(f(u_g) == v_h, f(v_g) == u_h))   # assuming undirected graph
            for u_h, v_h in H.edges()
        ]))

    return f

We can test our solver with two graphs that we know are isomorphic:

In [ ]:
s = Solver()
G = nx.complete_graph(3)
H = nx.complete_graph(3)

f = find_isomorphism(G, H, s)

print(s.check()) # should print sat
m = s.model()
print(m)
plot_isomorphism(G, H, f, m)

And two graphs that we know are not isomorphic:

In [ ]:
s = Solver()
G = nx.complete_graph(5)
H = nx.complete_graph(5)
H.remove_edges_from([(0, 1), (1, 2), (2, 3), (3, 4), (4, 0)])

f = find_isomorphism(G, H, s)

print(s.check()) # should print unsat

Now we can find the isomorphism between the two graphs from before:

In [ ]:
s = Solver()
G = nx.Graph()
G.add_nodes_from([1, 8])
edges = [(1, 2), (1, 4), (1, 6),
         (2, 3), (2, 5), (3, 4),
         (3, 8), (4, 7), (5, 6),
         (5, 8), (6, 7), (7, 8)]
G.add_edges_from(edges)
G_pos = {1: (0, 3), 2: (1, 3), 3: (0, 2), 4: (1, 2), 5: (0, 1), 6: (1, 1), 7: (0, 0), 8: (1, 0)}

H = nx.Graph()
H.add_nodes_from([1, 8])
edges = [(1, 2), (1, 4), (1, 5),
         (2, 3), (2, 6), (3, 4),
         (3, 7), (4, 8), (5, 6),
         (5, 8), (6, 7), (7, 8)]
H.add_edges_from(edges)
H_pos = {1: (0, 3), 2: (3, 3), 3: (3, 0), 4: (0, 0), 5: (1, 2), 6: (2, 2), 7: (2, 1), 8: (1, 1)}

f = find_isomorphism(G, H, s)

print(s.check()) # should print sat
m = s.model()
print(m)
plot_isomorphism(G, H, f, m, G_pos=G_pos, H_pos=H_pos)


###Congratulations! You just used an SMT solver to decide whether two graphs are isomorphic!


####If you'd like to continue your Z3 journey, you can start with this guide to learn more:
https://ericpony.github.io/z3py-tutorial/guide-examples.htm